In [1]:
import pandas as pd
import pypsa

In [2]:
n = pypsa.Network()

In [3]:
n.add("Bus", "Copperbelt", y=-15.42, x=28.2, v_nom=400)
n.add("Bus", "Lusaka", y=-13.0, x=28.0, v_nom=400)

In [4]:
n.add(
    "Line",
    "Lusaka-Copperbelt",
    bus0="Lusaka",
    bus1="Copperbelt",
    s_nom=1_500,
    x=1,
    r=1,
)

In [5]:
n.explore()

INFO:pypsa.plot:Components rendered on the map: Bus, Line.
INFO:pypsa.plot:Components omitted as they are missing or not selected: Generator, Link, Load, StorageUnit, Transformer.


In [6]:
n.add(
    "Carrier",
    ["coal", "hydro"],
    #co2_emissions=emissions,
    nice_name=["Coal", "Hydro"],
    color=["dimgrey", "royalblue"],
)

n.add("Carrier", "AC", nice_name="Electricity", color="crimson")

### Add generators

In [7]:
n.add(
    "Generator",
    "Hydro Generation",
    bus="Lusaka",
    carrier="hydro",
    p_nom=5000,  # MW
    p_max_pu=0.4,
    marginal_cost=0,  # default
)

In [8]:
fuel_cost = dict(
    coal=8,
    gas=100,
    oil=48,
)

In [9]:
efficiency = dict(
    coal=0.33,
    gas=0.58,
    oil=0.35,
)

In [10]:
n.add(
    "Generator",
    "Coal Generation",
    bus="Copperbelt",
    carrier="coal",
    efficiency=efficiency.get("coal", 1),
    p_nom=1750,
    marginal_cost=fuel_cost.get("coal", 0) / efficiency.get("coal", 1),
)
n.add(
    "Generator",
    "Oil Generation",
    bus="Copperbelt",
    carrier="oil",
    efficiency=efficiency.get("oil", 1),
    p_nom=1200,
    marginal_cost=fuel_cost.get("oil", 0) / efficiency.get("oil", 1),
)

### Add loads

In [11]:
loads = {
    "Lusaka": 700,
    "Copperbelt": 3000,
}

In [12]:
n.add(
    "Load",
    "Lusaka electricity demand",
    bus="Lusaka",
    p_set=loads["Lusaka"],
    carrier="AC",
)

In [13]:
n.add(
    "Load",
    "Copperbelt electricity demand",
    bus="Copperbelt",
    p_set=loads["Copperbelt"],
    carrier="AC",
)

In [14]:
n_dry_year = n.copy()
n_wet_year = n.copy()

In [15]:
n.optimize(solver_name="highs", log_to_console=False)

Index(['Hydro Generation', 'Coal Generation', 'Oil Generation'], dtype='object', name='Generator')
Index(['Hydro Generation', 'Coal Generation', 'Oil Generation'], dtype='object', name='Generator')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.01s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 4 primals, 10 duals
Objective: 4.12e+04
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.


Running HiGHS 1.11.0 (git hash: n/a): Copyright (c) 2025 HiGHS under MIT licence terms


('ok', 'optimal')

In [16]:
loads

{'Lusaka': 700, 'Copperbelt': 3000}

In [17]:
n.statistics()

Optimal Capacity  Installed Capacity  Supply  \
Generator coal                   1750.0              1750.0  1700.0   
          hydro                  5000.0              5000.0  2000.0   
          oil                    1200.0              1200.0     0.0   
Line      Electricity            1500.0              1500.0  1300.0   
Load      Electricity               0.0                 0.0     0.0   

                       Withdrawal  Energy Balance  Transmission  \
Generator coal                0.0          1700.0           0.0   
          hydro               0.0          2000.0           0.0   
          oil                 0.0             0.0           0.0   
Line      Electricity      1300.0             0.0        1300.0   
Load      Electricity      3700.0         -3700.0           0.0   

                       Capacity Factor  Curtailment  Capital Expenditure  \
Generator coal                0.971429         50.0                  0.0   
          hydro               0.400000          0.0                  0.0   
          oil                 0.000000       1200.0                  0.0   
Line      Electricity         0.866667          0.0                  0.0   
Load      Electricity              NaN          0.0                  0.0   

                       Operational Expenditure      Revenue  Market Value  
Generator coal                     41212.12121  41212.12121     24.242424  
          hydro                        0.00000  48484.84848     24.242424  
          oil                          0.00000      0.00000      0.000000  
Line      Electricity                  0.00000      0.00000           NaN  
Load      Electricity                  0.00000 -89696.96970           NaN

## Modify network

In [18]:
n.lines.loc["Lusaka-Copperbelt", "s_nom"] = 750

In [19]:
n.optimize()

Index(['Hydro Generation', 'Coal Generation', 'Oil Generation'], dtype='object', name='Generator')
Index(['Hydro Generation', 'Coal Generation', 'Oil Generation'], dtype='object', name='Generator')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.01s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 4 primals, 10 duals
Objective: 1.11e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.


Running HiGHS 1.11.0 (git hash: n/a): Copyright (c) 2025 HiGHS under MIT licence terms
LP   linopy-problem-mi6h8h83 has 10 rows; 4 cols; 13 nonzeros
Coefficient ranges:
  Matrix [1e+00, 1e+00]
  Cost   [2e+01, 1e+02]
  Bound  [0e+00, 0e+00]
  RHS    [7e+02, 3e+03]
Presolving model
1 rows, 2 cols, 2 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve : Reductions: rows 0(-10); columns 0(-4); elements 0(-13) - Reduced to empty
Solving the original LP from the solution after postsolve
Model name          : linopy-problem-mi6h8h83
Model status        : Optimal
Objective value     :  1.1099567100e+05
P-D objective error :  1.3110285785e-16
HiGHS run time      :          0.00
Writing the solution to /private/var/folders/qn/vpndfm21795ckkq89np1ckp40000gn/T/linopy-solve-lmo88dja.sol


('ok', 'optimal')

In [20]:
n.statistics()

Optimal Capacity  Installed Capacity  Supply  \
Generator coal                   1750.0              1750.0  1750.0   
          hydro                  5000.0              5000.0  1450.0   
          oil                    1200.0              1200.0   500.0   
Line      Electricity             750.0               750.0   750.0   
Load      Electricity               0.0                 0.0     0.0   

                       Withdrawal  Energy Balance  Transmission  \
Generator coal                0.0          1750.0           0.0   
          hydro               0.0          1450.0           0.0   
          oil                 0.0           500.0           0.0   
Line      Electricity       750.0             0.0         750.0   
Load      Electricity      3700.0         -3700.0           0.0   

                       Capacity Factor  Curtailment  Capital Expenditure  \
Generator coal                1.000000          0.0                  0.0   
          hydro               0.290000        550.0                  0.0   
          oil                 0.416667        700.0                  0.0   
Line      Electricity         1.000000          0.0                  0.0   
Load      Electricity              NaN          0.0                  0.0   

                       Operational Expenditure       Revenue  Market Value  
Generator coal                     42424.24242  240000.00000    137.142857  
          hydro                        0.00000       0.00000           NaN  
          oil                      68571.42857   68571.42857    137.142857  
Line      Electricity                  0.00000  102857.14286    137.142857  
Load      Electricity                  0.00000 -411428.57143           NaN

## Hydrological scenarios

In [21]:
n_dry_year.generators.loc["Hydro Generation", "p_max_pu"] = 0.3
n_wet_year.generators.loc["Hydro Generation", "p_max_pu"] = 0.6

In [22]:
n_dry_year.optimize(solver_name="highs", log_to_console=False)

Index(['Hydro Generation', 'Coal Generation', 'Oil Generation'], dtype='object', name='Generator')
Index(['Hydro Generation', 'Coal Generation', 'Oil Generation'], dtype='object', name='Generator')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.01s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 4 primals, 10 duals
Objective: 1.04e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.


Running HiGHS 1.11.0 (git hash: n/a): Copyright (c) 2025 HiGHS under MIT licence terms


('ok', 'optimal')

In [23]:
n_wet_year.optimize(solver_name="highs", log_to_console=False)

Index(['Hydro Generation', 'Coal Generation', 'Oil Generation'], dtype='object', name='Generator')
Index(['Hydro Generation', 'Coal Generation', 'Oil Generation'], dtype='object', name='Generator')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.01s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 4 primals, 10 duals
Objective: 3.64e+04
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper were not assigned to the network.


Running HiGHS 1.11.0 (git hash: n/a): Copyright (c) 2025 HiGHS under MIT licence terms


('ok', 'optimal')

In [24]:
n.statistics()

Optimal Capacity  Installed Capacity  Supply  \
Generator coal                   1750.0              1750.0  1750.0   
          hydro                  5000.0              5000.0  1450.0   
          oil                    1200.0              1200.0   500.0   
Line      Electricity             750.0               750.0   750.0   
Load      Electricity               0.0                 0.0     0.0   

                       Withdrawal  Energy Balance  Transmission  \
Generator coal                0.0          1750.0           0.0   
          hydro               0.0          1450.0           0.0   
          oil                 0.0           500.0           0.0   
Line      Electricity       750.0             0.0         750.0   
Load      Electricity      3700.0         -3700.0           0.0   

                       Capacity Factor  Curtailment  Capital Expenditure  \
Generator coal                1.000000          0.0                  0.0   
          hydro               0.290000        550.0                  0.0   
          oil                 0.416667        700.0                  0.0   
Line      Electricity         1.000000          0.0                  0.0   
Load      Electricity              NaN          0.0                  0.0   

                       Operational Expenditure       Revenue  Market Value  
Generator coal                     42424.24242  240000.00000    137.142857  
          hydro                        0.00000       0.00000           NaN  
          oil                      68571.42857   68571.42857    137.142857  
Line      Electricity                  0.00000  102857.14286    137.142857  
Load      Electricity                  0.00000 -411428.57143           NaN

In [25]:
n_dry_year.statistics()

Optimal Capacity  Installed Capacity  Supply  \
Generator coal                   1750.0              1750.0  1750.0   
          hydro                  5000.0              5000.0  1500.0   
          oil                    1200.0              1200.0   450.0   
Line      Electricity            1500.0              1500.0   800.0   
Load      Electricity               0.0                 0.0     0.0   

                       Withdrawal  Energy Balance  Transmission  \
Generator coal                0.0          1750.0           0.0   
          hydro               0.0          1500.0           0.0   
          oil                 0.0           450.0           0.0   
Line      Electricity       800.0             0.0         800.0   
Load      Electricity      3700.0         -3700.0           0.0   

                       Capacity Factor  Curtailment  Capital Expenditure  \
Generator coal                1.000000          0.0                  0.0   
          hydro               0.300000          0.0                  0.0   
          oil                 0.375000        750.0                  0.0   
Line      Electricity         0.533333          0.0                  0.0   
Load      Electricity              NaN          0.0                  0.0   

                       Operational Expenditure       Revenue  Market Value  
Generator coal                     42424.24242  240000.00000    137.142857  
          hydro                        0.00000  205714.28571    137.142857  
          oil                      61714.28571   61714.28571    137.142857  
Line      Electricity                  0.00000       0.00000           NaN  
Load      Electricity                  0.00000 -507428.57143           NaN

In [26]:
n_wet_year.statistics()

Optimal Capacity  Installed Capacity  Supply  \
Generator coal                   1750.0              1750.0  1500.0   
          hydro                  5000.0              5000.0  2200.0   
          oil                    1200.0              1200.0     0.0   
Line      Electricity            1500.0              1500.0  1500.0   
Load      Electricity               0.0                 0.0     0.0   

                       Withdrawal  Energy Balance  Transmission  \
Generator coal                0.0          1500.0           0.0   
          hydro               0.0          2200.0           0.0   
          oil                 0.0             0.0           0.0   
Line      Electricity      1500.0             0.0        1500.0   
Load      Electricity      3700.0         -3700.0           0.0   

                       Capacity Factor  Curtailment  Capital Expenditure  \
Generator coal                0.857143        250.0                  0.0   
          hydro               0.440000        800.0                  0.0   
          oil                 0.000000       1200.0                  0.0   
Line      Electricity         1.000000          0.0                  0.0   
Load      Electricity              NaN          0.0                  0.0   

                       Operational Expenditure      Revenue  Market Value  
Generator coal                     36363.63636  36363.63636     24.242424  
          hydro                        0.00000      0.00000           NaN  
          oil                          0.00000      0.00000      0.000000  
Line      Electricity                  0.00000  36363.63636     24.242424  
Load      Electricity                  0.00000 -72727.27273           NaN